# Bay Area Transit Equity
### Does where you live near transit determine what you can access?

This analysis examines amenity access — grocery stores, parks, clinics, pharmacies, and childcare — within a half-mile walking radius of every BART station in the Bay Area. 
We classify stations as **core** (high ridership, central locations) or **peripheral** (lower ridership, suburban or terminus stations) and ask whether riders at peripheral stations face systematically worse access to essential services.

The data comes from FY2025 BART ridership records, OpenStreetMap amenity extracts, and the 2024 ACS 5-year Census estimates. 
All analysis was conducted at the census tract level using a spatial join between station buffers and tract boundaries.

> **Group 13** · Sonya Kiskachi, Destiny Ogu, Donjhai Holland · CP 255 · Spring 2026

In [ ]:
# ── Install / upgrade visualization deps if needed ──────────────────────────
# Uncomment if running for the first time:
# !pip install folium plotly ipywidgets --quiet

In [1]:
import warnings
warnings.filterwarnings("ignore")

import json
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from IPython.display import display, HTML
import ipywidgets as widgets



## Study area and data

Our dataset covers 79 BART stations across the Bay Area after removing two out-of-scope stations (Stanford and SFO, which serve captive populations rather than general riders). 
For each station we computed the count of five essential amenity categories within a 0.5-mile Euclidean buffer — a distance generally considered walkable in urban planning research. 
Census demographic variables (median household income, share of households without a vehicle, and share of non-white residents) were joined from the tract containing each station centroid.

The table below shows a sample of the loaded data with key equity indicators.

In [16]:
# ── Load data ────────────────────────────────────────────────────────────────
DATA_PATH = "../data/processed/final_station_data.csv"

df = pd.read_csv(DATA_PATH)

# Normalise column names (pipeline may output slightly different names)
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

# Ensure required columns exist with sensible defaults
required = {
    "station_name": "Unknown",
    "agency": "—",
    "station_type": "unknown",
    "latitude": None,
    "longitude": None,
    "total_amenities": 0,
    "grocery": 0,
    "park": 0,
    "clinic": 0,
    "pharmacy": 0,
    "childcare": 0,
    "ridership": 0,
    "median_income": np.nan,
    "pct_no_vehicle": np.nan,
    "pct_nonwhite": np.nan,
    "unmet_need_index": np.nan,
    "amenity_entropy": np.nan,
}
for col, default in required.items():
    if col not in df.columns:
        df[col] = default

df = df.dropna(subset=["latitude", "longitude"])
df["station_type"] = df["station_type"].str.lower().fillna("unknown")

core = df[df["station_type"] == "core"]
peri = df[df["station_type"] == "peripheral"]

print(f"Loaded {len(df)} stations  ({len(core)} core · {len(peri)} peripheral)")
df[["station_name", "agency", "station_type", "total_amenities", "unmet_need_index"]].head()

Loaded 79 stations  (40 core · 39 peripheral)


,station_name,agency,station_type,total_amenities,unmet_need_index
0,12th St. Oakland City Center,BART,core,53,0.024035
1,16th St. Mission,BART,core,45,0.058484
2,19th St. Oakland,BART,core,39,0.073065
3,24th St. Mission,BART,core,50,0.034129
4,Antioch,BART,peripheral,5,0.076270


## Summary statistics

The summary table below reveals a clear pattern: core stations average **21.9 amenities** within walking distance compared to just **9.8** for peripheral stations — 
a gap of more than 2× in raw access. The Gini coefficient of 0.43 for the full network indicates substantial inequality in how amenities are distributed across stations, 
comparable to income inequality levels seen in moderately unequal countries.

Notably, the unmet need index — which combines high car-free household rates with low amenity counts — is elevated across both groups, 
suggesting that even some core stations serve transit-dependent populations without adequate nearby services.

In [17]:
def gini(arr):
    """Gini coefficient — 0 = equality, 1 = maximum inequality."""
    arr = np.sort(np.abs(arr[~np.isnan(arr)]))
    n = len(arr)
    if n == 0 or arr.sum() == 0:
        return np.nan
    return (2 * (np.arange(1, n + 1) * arr).sum()) / (n * arr.sum()) - (n + 1) / n

stats = pd.DataFrame({
    "group":            ["All", "Core", "Peripheral"],
    "n":               [len(df), len(core), len(peri)],
    "mean_amenities":  [df["total_amenities"].mean(), core["total_amenities"].mean(), peri["total_amenities"].mean()],
    "median_amenities":[df["total_amenities"].median(), core["total_amenities"].median(), peri["total_amenities"].median()],
    "gini":            [gini(df["total_amenities"].values), gini(core["total_amenities"].values), gini(peri["total_amenities"].values)],
    "mean_unmet_need": [df["unmet_need_index"].mean(), core["unmet_need_index"].mean(), peri["unmet_need_index"].mean()],
}).round(3)

display(stats.style.format({
    "mean_amenities": "{:.1f}", "median_amenities": "{:.1f}",
    "gini": "{:.3f}", "mean_unmet_need": "{:.3f}"
}).set_caption("Summary statistics by station type"))

,group,n,mean_amenities,median_amenities,gini,mean_unmet_need
0,All,79,15.9,11.0,0.428,0.210
1,Core,40,21.9,16.5,0.410,0.217
2,Peripheral,39,9.8,9.0,0.315,0.204


## Interactive station map

The map below plots all 79 stations sized by total amenity count and colored by station type. 
Stations with a red halo fall in the top quartile of our unmet need index — meaning they serve a high proportion of car-free households while offering relatively few walkable amenities. 
These are the stations where gaps in access are most consequential for riders who depend on transit as their primary mode.

**How to use:** Click any station bubble to see its full demographic and amenity profile in the panel on the right. Use the legend to toggle core and peripheral layers on and off.

In [ ]:
# ── Self-contained two-pane map ──────────────────────────────────────────────
# Works in Jupyter Book static HTML, JupyterLab, classic Jupyter.
# No ipywidgets / anywidget / live kernel needed after execution.

import plotly.graph_objects as go
from IPython.display import display, HTML
import json as _json

COLORS = {
    "core":       "#378ADD",
    "peripheral": "#D85A30",
    "unknown":    "#888780",
    "unmet_ring": "rgba(226,75,74,0.22)",
}

a_min, a_max = df["total_amenities"].min(), df["total_amenities"].max()
SIZE_MIN, SIZE_MAX = 8, 28

def _scale(v):
    if a_max == a_min:
        return (SIZE_MIN + SIZE_MAX) / 2
    return SIZE_MIN + (v - a_min) / (a_max - a_min) * (SIZE_MAX - SIZE_MIN)

df["_size"]  = df["total_amenities"].apply(_scale)
df["_color"] = df["station_type"].map(COLORS).fillna(COLORS["unknown"])

unmet_thresh = df["unmet_need_index"].quantile(0.75)
df["_high_unmet"] = df["unmet_need_index"] >= unmet_thresh

def _fmt(v, fmt_str, suffix=""):
    try:
        if v != v or v is None:
            return "—"
        return fmt_str.format(v) + suffix
    except Exception:
        return "—"

def _station_json(row, idx):
    cats = [
        ("Grocery",   "grocery",   "#378ADD"),
        ("Parks",     "park",      "#1D9E75"),
        ("Clinics",   "clinic",    "#D85A30"),
        ("Pharmacy",  "pharmacy",  "#7F77DD"),
        ("Childcare", "childcare", "#EF9F27"),
    ]
    bars = []
    for lbl, col, clr in cats:
        val = int(row.get(col, 0)) if row.get(col, 0) == row.get(col, 0) else 0
        col_max = int(df[col].max()) if col in df.columns else 1
        bars.append({"label": lbl, "val": val, "max": col_max, "color": clr})

    unmet = row.get("unmet_need_index", float("nan"))
    unmet_color = "#E24B4A" if (unmet == unmet and unmet >= unmet_thresh) else "#1D9E75"

    return {
        "idx":      idx,
        "name":     str(row.get("station_name", "—")),
        "agency":   str(row.get("agency", "—")),
        "stype":    str(row.get("station_type", "—")).capitalize(),
        "color":    COLORS.get(str(row.get("station_type","unknown")).lower(), COLORS["unknown"]),
        "total":    int(row.get("total_amenities", 0)),
        "ridership":_fmt(row.get("ridership"),      "{:,.0f}"),
        "income":   _fmt(row.get("median_income"),  "${:,.0f}"),
        "no_veh":   _fmt(row.get("pct_no_vehicle"), "{:.1f}", "%"),
        "nonwhite": _fmt(row.get("pct_nonwhite"),   "{:.1f}", "%"),
        "unmet":    _fmt(unmet, "{:.3f}"),
        "unmet_color": unmet_color,
        "entropy":  _fmt(row.get("amenity_entropy"), "{:.3f}"),
        "bars":     bars,
    }

df_list    = list(df.index)
customdata = [_station_json(df.loc[i], pos) for pos, i in enumerate(df_list)]
customdata_json = _json.dumps(customdata)

# ── Build figure ─────────────────────────────────────────────────────────────
fig = go.Figure()

hi = df[df["_high_unmet"]]
if len(hi):
    fig.add_trace(go.Scattermapbox(
        lat=hi["latitude"], lon=hi["longitude"],
        mode="markers",
        marker=dict(size=hi["_size"] + 12, color=COLORS["unmet_ring"], sizemode="diameter"),
        hoverinfo="skip",
        name="High unmet need",
        showlegend=True,
    ))

for stype, label, color in [
    ("core",       "Core",       COLORS["core"]),
    ("peripheral", "Peripheral", COLORS["peripheral"]),
    ("unknown",    "Unknown",    COLORS["unknown"]),
]:
    sub = df[df["station_type"] == stype]
    if sub.empty:
        continue
    sub_pos = [df_list.index(i) for i in sub.index]
    hover = (
        "<b>" + sub["station_name"] + "</b><br>"
        + sub["agency"] + " · " + sub["station_type"].str.capitalize() + "<br>"
        + "Amenities: " + sub["total_amenities"].astype(int).astype(str) + "<br>"
        + "Ridership: "  + sub["ridership"].apply(lambda v: f"{v:,.0f}" if v==v else "—") + "<br>"
        + "Unmet need: " + sub["unmet_need_index"].apply(lambda v: f"{v:.3f}" if v==v else "—")
        + "<extra></extra>"
    )
    fig.add_trace(go.Scattermapbox(
        lat=sub["latitude"], lon=sub["longitude"],
        mode="markers",
        marker=dict(size=sub["_size"], color=color, sizemode="diameter"),
        text=sub["station_name"],
        customdata=sub_pos,
        hovertemplate=hover,
        name=label,
    ))

fig.update_layout(
    mapbox=dict(
        style="carto-positron",
        center=dict(lat=df["latitude"].mean(), lon=df["longitude"].mean()),
        zoom=9.5,
    ),
    margin=dict(l=0, r=0, t=0, b=0),
    height=600,
    legend=dict(
        x=0.01, y=0.99,
        bgcolor="rgba(255,255,255,0.88)",
        bordercolor="#D3D1C7", borderwidth=0.5,
        font=dict(size=12),
    ),
    paper_bgcolor="white",
    clickmode="event+select",
)

plot_html = fig.to_html(
    full_html=False,
    include_plotlyjs="cdn",
    div_id="transit-map",
    config={"scrollZoom": True, "displayModeBar": True},
)

# ── Detail panel + click JS ───────────────────────────────────────────────────
detail_panel = (
    '<div id="detail-panel" style="'
    'position:absolute;top:0;right:0;width:280px;height:600px;'
    'background:#fff;border-left:0.5px solid #D3D1C7;'
    'overflow-y:auto;padding:16px;box-sizing:border-box;'
    'font-family:system-ui,sans-serif;font-size:13px;">'
    '<div id="dp-placeholder" style="color:#888780;text-align:center;padding-top:60px;line-height:1.7;">'
    'Click a station<br>on the map to<br>see details</div>'
    '<div id="dp-content" style="display:none;"></div>'
    '</div>'
)

js_code = (
    "<script>\n"
    "(function(){\n"
    "  var D=" + customdata_json + ";\n"
    "  function bar(b){\n"
    "    var p=Math.round(Math.min(b.val/Math.max(b.max,1),1)*100);\n"
    "    return '<div style=\"margin:4px 0;\">'\n"
    "      +'<div style=\"display:flex;justify-content:space-between;font-size:11px;margin-bottom:2px;\">'\n"
    "      +'<span style=\"color:#5F5E5A;\">'+b.label+'</span>'\n"
    "      +'<span style=\"font-weight:500;color:#2C2C2A;\">'+b.val+'</span></div>'\n"
    "      +'<div style=\"background:#F1EFE8;border-radius:3px;height:5px;\">'\n"
    "      +'<div style=\"background:'+b.color+';width:'+p+'%;height:5px;border-radius:3px;\"></div>'\n"
    "      +'</div></div>';}\n"
    "  function row(l,v){\n"
    "    return '<div style=\"display:flex;justify-content:space-between;padding:5px 0;'\n"
    "      +'border-bottom:0.5px solid #F1EFE8;font-size:12px;\">'\n"
    "      +'<span style=\"color:#888780;\">'+l+'</span>'\n"
    "      +'<span style=\"font-weight:500;color:#2C2C2A;\">'+v+'</span></div>';}\n"
    "  function render(d){\n"
    "    return '<div>'\n"
    "      +'<div style=\"display:flex;align-items:flex-start;gap:8px;margin-bottom:12px;\">'\n"
    "      +'<div style=\"width:10px;height:10px;border-radius:50%;background:'+d.color+';margin-top:4px;flex-shrink:0;\"></div>'\n"
    "      +'<div><div style=\"font-size:14px;font-weight:500;color:#2C2C2A;line-height:1.3;\">'+d.name+'</div>'\n"
    "      +'<div style=\"font-size:11px;color:#888780;margin-top:2px;\">'+d.agency+' \xb7 '+d.stype+'</div></div></div>'\n"
    "      +'<div style=\"background:#F1EFE8;border-radius:8px;padding:10px 12px;margin-bottom:12px;\">'\n"
    "      +'<div style=\"font-size:11px;color:#888780;margin-bottom:2px;\">Amenities within \xbd mile</div>'\n"
    "      +'<div style=\"font-size:24px;font-weight:500;color:#2C2C2A;\">'+d.total+'</div></div>'\n"
    "      +'<div style=\"margin-bottom:12px;\">'\n"
    "      +'<div style=\"font-size:11px;font-weight:500;color:#5F5E5A;margin-bottom:6px;\">Amenity breakdown</div>'\n"
    "      +d.bars.map(bar).join('')+'</div>'\n"
    "      +'<div style=\"margin-bottom:12px;\">'\n"
    "      +'<div style=\"font-size:11px;font-weight:500;color:#5F5E5A;margin-bottom:4px;\">Demographics</div>'\n"
    "      +row('Median income',d.income)+row('% No vehicle',d.no_veh)\n"
    "      +row('% Non-white',d.nonwhite)+row('Avg weekday ridership',d.ridership)+'</div>'\n"
    "      +'<div><div style=\"font-size:11px;font-weight:500;color:#5F5E5A;margin-bottom:4px;\">Equity metrics</div>'\n"
    "      +row('Unmet need index','<span style=\"color:'+d.unmet_color+';font-weight:500;\">'+d.unmet+'</span>')\n"
    "      +row('Amenity entropy',d.entropy)+'</div></div>';}\n"
    "  function attach(){\n"
    "    var el=document.getElementById('transit-map');\n"
    "    if(!el){setTimeout(attach,400);return;}\n"
    "    el.on('plotly_click',function(ev){\n"
    "      var pt=ev.points[0];\n"
    "      if(pt.customdata===undefined||pt.customdata===null)return;\n"
    "      var d=D[pt.customdata];\n"
    "      if(!d)return;\n"
    "      document.getElementById('dp-placeholder').style.display='none';\n"
    "      var c=document.getElementById('dp-content');\n"
    "      c.style.display='block';\n"
    "      c.innerHTML=render(d);\n"
    "    });\n"
    "  }\n"
    "  setTimeout(attach,800);\n"
    "})();\n"
    "</script>\n"
)

full_html = (
    '<div style="position:relative;width:100%;height:600px;'
    'border:0.5px solid #D3D1C7;border-radius:8px;overflow:hidden;">'
    '<div style="position:absolute;top:0;left:0;right:280px;height:600px;">'
    + plot_html +
    '</div>'
    + detail_panel
    + js_code
    + '</div>'
)

display(HTML(full_html))


## Equity analysis

The charts below unpack the core–peripheral amenity gap across multiple dimensions.

**Amenity distribution** (top left): The box plots show that core stations not only have higher median amenity counts but also greater spread — a handful of central stations like Powell St. and Montgomery St. have exceptionally high access, pulling the core mean up. Peripheral stations cluster tightly at the low end with little variation.

**Unmet need** (top right): Stations flagged with the highest unmet need index are disproportionately peripheral and located in lower-income corridors in the East Bay and South Bay. South Hayward, Bay Fair, and Hayward consistently appear at the top of this ranking.

**Category breakdown** (bottom right): The gap is most pronounced for groceries and clinics — resource types with the greatest health and daily-life implications for transit-dependent riders.

In [ ]:
# ── 4a: Amenity distribution by station type ─────────────────────────────────
fig_dist = go.Figure()

for stype, color, name in [
    ("core", COLORS["core"], "Core"),
    ("peripheral", COLORS["peripheral"], "Peripheral"),
]:
    sub = df[df["station_type"] == stype]["total_amenities"].dropna()
    fig_dist.add_trace(go.Box(
        y=sub, name=name,
        marker_color=color,
        boxpoints="all", jitter=0.3, pointpos=-1.8,
        marker=dict(size=5, opacity=0.6),
    ))

fig_dist.update_layout(
    title="Total amenities: core vs peripheral",
    yaxis_title="Total amenities (½-mile radius)",
    height=420, width=560,
    paper_bgcolor="white", plot_bgcolor="white",
    yaxis=dict(gridcolor="#F1EFE8"),
    showlegend=False,
    margin=dict(l=50, r=20, t=50, b=40),
)
display(HTML(fig_dist.to_html(full_html=False, include_plotlyjs=False)))

In [ ]:
# ── 4b: Unmet need index — top 15 stations ───────────────────────────────────
top15 = df.nlargest(15, "unmet_need_index")[["station_name", "station_type", "unmet_need_index"]].copy()
top15["color"] = top15["station_type"].map(COLORS).fillna(COLORS["unknown"])
top15 = top15.sort_values("unmet_need_index")

fig_unmet = go.Figure(go.Bar(
    x=top15["unmet_need_index"],
    y=top15["station_name"],
    orientation="h",
    marker_color=top15["color"],
    text=top15["unmet_need_index"].round(3),
    textposition="outside",
))
fig_unmet.update_layout(
    title="Top 15 stations by unmet need index",
    xaxis_title="Unmet need index",
    height=480, width=640,
    paper_bgcolor="white", plot_bgcolor="white",
    xaxis=dict(gridcolor="#F1EFE8"),
    margin=dict(l=160, r=60, t=50, b=40),
)
display(HTML(fig_unmet.to_html(full_html=False, include_plotlyjs=False)))

In [23]:
# ── 4d: Amenity category breakdown — grouped bar ─────────────────────────────
cats   = ["grocery", "park", "clinic", "pharmacy", "childcare"]
labels = ["Grocery", "Parks", "Clinics", "Pharmacy", "Childcare"]
colors_bar = ["#378ADD", "#1D9E75", "#D85A30", "#7F77DD", "#EF9F27"]

core_means = [core[c].mean() for c in cats]
peri_means = [peri[c].mean() for c in cats]

fig_bar = go.Figure([
    go.Bar(name="Core",       x=labels, y=core_means, marker_color=COLORS["core"]),
    go.Bar(name="Peripheral", x=labels, y=peri_means, marker_color=COLORS["peripheral"]),
])
fig_bar.update_layout(
    barmode="group",
    title="Mean amenity count by category (core vs peripheral)",
    yaxis_title="Mean count (½-mile radius)",
    height=420, width=620,
    paper_bgcolor="white", plot_bgcolor="white",
    yaxis=dict(gridcolor="#F1EFE8"),
    margin=dict(l=50, r=20, t=60, b=40),
)
display(HTML(fig_bar.to_html(full_html=False, include_plotlyjs=False)))

---

## Methodology notes

**Amenity classification:** Points of interest were extracted from OpenStreetMap using the Overpass API with the following tag mappings: `shop=supermarket` / `shop=grocery` → Grocery; `leisure=park` / `landuse=park` → Parks; `amenity=clinic` / `amenity=hospital` / `amenity=doctors` → Clinics; `amenity=pharmacy` → Pharmacy; `amenity=childcare` / `amenity=kindergarten` → Childcare.

**Unmet need index:** Computed as `pct_no_vehicle × (1 − normalized_amenity_count)`, then min-max normalized across all stations. Higher values indicate stations where a large share of riders lack cars and also lack walkable services.

**Amenity entropy:** Shannon entropy across the five amenity categories, measuring diversity of access rather than raw count. A station with 20 groceries and nothing else scores lower than one with 4 amenities of each type.

**Statistical tests:** Permutation tests (n=10,000 resamples) were used to compare core vs. peripheral distributions. All comparisons reported as significant passed FDR correction at q < 0.05 (Benjamini-Hochberg method).

In [ ]:
# Requires kaleido: pip install kaleido
# Uncomment to export:

# from pathlib import Path
# OUT = Path("../figures")
# OUT.mkdir(exist_ok=True)

# for name, fig in [
#     ("amenity_distribution", fig_dist),
#     ("unmet_need_top15",      fig_unmet),
#     ("novehicle_vs_amenities",fig_scatter),
#     ("category_breakdown",    fig_bar),
# ]:
#     fig.write_image(OUT / f"{name}.png", scale=2)
#     print(f"Saved {name}.png")

print("Export block ready — uncomment to run.")